In [2]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
import os

In [3]:
load_dotenv(override=True)  # override=True 确保 .env 会覆盖系统已有的同名变量
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))  # 确认代理是否生效

HTTPS_PROXY: http://127.0.0.1:7897


In [4]:
# 不知道什么现在加temperature会超时 后面研究下吧
# model = init_chat_model(
#     "openrouter:deepseek/deepseek-v4-flash-0731",
# )
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    # temperature=0.5,    # 控制模型输出的随机性。数值越高，响应越具创造性
    # timeout = 30,       # 超时时间
    # max_retries = 3,    # 最大重试次数
)

In [4]:
# 正常数组消息
# conversation = [
#     {"role":"system","content":"You are a helpful assistant that translates English to French."},
#     {"role": "user", "content": "Translate: I love programming."},
#     {"role": "assistant", "content": "J'adore la programmation."},
#     {"role": "user", "content": "Translate: I love building applications."}
# ]
conversation =[
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Translate: I love programming."),
    AIMessage(content="J'adore la programmation."),
    HumanMessage(content="Translate: I love building applications."),
]

In [5]:
# 流式输出
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

# 流式输出 每次输出都会增加
full = None
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)
print(full.content_blocks)

|||||||||||||||||||||||||||||||||||||||||||||||||||||Par|rots’ colorful feathers are the| result of millions of years of evolution|, and they serve| several important biological purposes|:

### 1. **|Camouflage —| blending into the rainforest|**
In the wild|, many parrots are mostly| **green**, which makes| them nearly invisible among| leaves and canopy shadows|. The green comes| from a combination of yellow| pigments and blue structural colors|, created by tiny| structures in the feathers that| reflect light. This| helps them hide from predators like| hawks and snakes|.

### 2|. **Communication and| social bonding**
Parrots are| highly social birds.| Bright colors help them recognize| each other, signal| mood, and maintain| pair bonds. Many parrots form| lifelong monogamous partnerships|, and colorful plumage may| help individuals identify their| mate from a distance|.

### 3|. **Sexual| selection and mate attraction|**
In many birds|, males are flash|ier to attract females. Par|rots 

In [ ]:
# batch 需要注意返回顺序时调用
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
# batch_as_completed 不需要注意返回顺序 谁先返回谁先调用
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)
for response in responses:
    print(response)

response = model.invoke(conversation)
print(response)

In [5]:
from langchain.tools import tool
@tool
def get_weather(city:str) -> str:
    """Get the weather at a location."""
    return f"The weather of {city} is sunny"

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
print(response)

Tool: get_weather
Args: {'city': 'Boston'}
content='' additional_kwargs={'reasoning_content': "The user asks for weather in Boston. I'll call get_weather.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user asks for weather in Boston. I'll call get_weather."}]} response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1787129268-Jl0QSxeb2whGrbv46PHe', 'created': 1787129268, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 4.284e-05, 'cost_details': {'upstream_inference_completions_cost': 1.281e-05, 'upstream_inference_prompt_cost': 3.003e-05, 'upstream_inference_cost': 4.284e-05}} id='lc_run--01a01934-85a4-7492-8f73-c4b513b7ed5b-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_c255fa15615c41678111290e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 286, 'output_tokens': 61, 'total_tokens':

In [ ]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str =Field(description = "The title of the movie")
    year:int = Field(description = "The year the movie was released")
    director:str = Field(description = "The director of the movie")
    rating:float = Field(description = "The rating of the movie")

model_with_movic = model.with_structured_output(Movie)
response = model_with_movic.invoke("What is the rating of the movie 'The Dark Knight'?")
print(response)
print(response.usage_metadata)

NameError: name 'model' is not defined